# Notatnik 09 — Statistical Significance & LPPLS Ablation Study

**Cel:** ocenić, czy dodanie cech LPPLS zwiększa zdolność uszeregowania dni według ryzyka przyszłego spadku BTC, a także które grupy cech LPPLS są użyteczne.

**Dwie części:**
1. **Porównanie prognoz z 08b** (bez ponownego trenowania): sparowane różnice Average Precision (AP), wskaźnik przewagi nad częstością klasy pozytywnej, wyniki roczne i bootstrap blokowy.
2. **Ablation study** (nowe dopasowania): pięć zagnieżdżonych zestawów cech, identyczne okresy treningowe, walidacyjne i testowe. Dla uczciwej kontrolowanej ablacji trzymamy **te same hiperparametry w obrębie modelu i foldu**, pobrane z wyniku strojenia hybrydy 08b.

**Ograniczenia naukowe:** na podstawie wcześniej oglądanych lat 2021–2024 nie mamy nowego, niewykorzystywanego zbioru testowego. Bootstrap blokowy daje przybliżoną ocenę niepewności warunkową na istniejących prognozach i założonym rozmiarze bloku. To **nie jest** automatycznie formalny dowód istotności ani korekta za wielokrotne porównania.

**Pliki wejściowe z notatnika 08b:**
- `wyniki_tuning_08b/08b_04_predykcje_OOS.csv` — wymagany dla części 1,
- `wyniki_tuning_08b/01_cechy_dzienne.csv` oraz `wyniki_tuning_08b/08b_02_wybrane_hiperparametry.csv` — wymagane dla części 2.

Jeśli plików nie ma, uruchom najpierw 08b z `RUN_ALL_NINE=True`, aby objąć wszystkie 9 konfiguracji.

In [ ]:
import sys as _sys
from pathlib import Path as _ResearchPath
_sys.path.insert(0,str(_ResearchPath('.').resolve()/'src'))
from lppls_research import TemporalAvailabilityValidator
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import (
    average_precision_score, roc_auc_score, confusion_matrix, f1_score
)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

SEED = 42
ROOT = Path('.')
BASE08 = ROOT / '08b_wyniki_tuning'
OUT = ROOT / '09_wyniki_ablation'
OUT.mkdir(parents=True, exist_ok=True)

# Część 1 — bootstrap blokowy
BOOTSTRAP_REPS = 600  # próba techniczna: 100; publikacja: np. 2000–5000 po analizie stabilności
BLOCK_SIZE = 60       # horyzont maksymalny, analiza wrażliwości 30/60/90
BLOCK_SENSITIVITY = (30, 60, 90)
MIN_POSITIVES_FOR_AP = 1

# Część 2 — kontrolowana ablation study
RUN_ABLATION = True
ABLATION_MODELS = ['LogisticRegression', 'DecisionTree', 'XGBoost', 'LightGBM', 'CatBoost']
# Po teście możesz użyć pełnej listy:
# ['LogisticRegression', 'DecisionTree', 'XGBoost', 'LightGBM', 'CatBoost']
ABLATION_TARGETS = [(d, h) for h in (30,45,60) for d in (10,15,20)]
# Po teście: [(d, h) for h in (30,45,60) for d in (10,15,20)]
PROB_THRESHOLDS = np.round(np.arange(0.10, 0.91, 0.05), 2)


def locate(filename):
    locations = [BASE08 / filename, ROOT / filename]
    for p in locations:
        if p.is_file():
            return p
    raise FileNotFoundError(
        f'Brak pliku {filename}. Szukano: {locations}. '
        'Uruchom notatnik 08b i zapisz jego wyniki w wyniki_tuning_08b.'
    )

print('Folder wynikowy:', OUT.resolve())
print('Modele ablation:', ABLATION_MODELS)
print('Zadania ablation:', ABLATION_TARGETS)

## 1. Wczytanie i kontrola prognoz OOS z 08b

Wszystkie porównania będą wykonywane **na tych samych dniach testowych**. Nie porównujemy AP z dwóch modeli, jeśli były oceniane na innych zestawach dni.

In [ ]:
path_preds = locate('08b_04_predykcje_OOS.csv')
preds = pd.read_csv(path_preds, parse_dates=['date'])
required = {'date','year','drop_pct','horizon','model','feature_group','y_true','score'}
missing = required - set(preds.columns)
if missing:
    raise ValueError(f'Brak kolumn w 08b_04_predykcje_OOS.csv: {missing}')
preds['date'] = pd.to_datetime(preds['date'], utc=True).dt.tz_convert(None).dt.normalize()
for c in ('year','drop_pct','horizon','y_true'):
    preds[c] = pd.to_numeric(preds[c], errors='raise').astype(int)
preds['score'] = pd.to_numeric(preds['score'], errors='raise')
assert preds['y_true'].isin([0,1]).all()
assert preds['score'].between(0,1).all()
assert preds['score'].notna().all()
key = ['date','drop_pct','horizon','model','feature_group']
assert not preds.duplicated(key).any(), 'Powtarzające się prognozy: najpierw usuń przyczynę duplikatów.'
assert preds['date'].dt.year.eq(preds['year']).all()
print('Predykcje:', len(preds))
print('Konfiguracje:', sorted(set(zip(preds['drop_pct'], preds['horizon']))))
print('Dostępne algorytmy:', sorted(preds['model'].unique()))
display(preds.head(3))

## 2. Narzędzia: dopasowanie prognoz i bootstrap blokowy

Losujemy **całe bloki kolejnych dni** z osobna w każdym roku testowym. Ponowne próbkowanie paruje predykcje obu modeli na tych samych datach, co pozwala mierzyć różnicę AP. Nie łączymy bloków przez granice lat.

Współczynnik `positive_rate` jest podstawowym punktem odniesienia dla AP. Raportujemy więc również `AP_minus_prevalence`.

In [ ]:
def matched_predictions(frame, drop, horizon, model, group_a, group_b):
    subset = frame.loc[
        frame.drop_pct.eq(drop) & frame.horizon.eq(horizon) & frame.model.eq(model)
    ]
    a = subset.loc[subset.feature_group.eq(group_a), ['date','year','y_true','score']].rename(
        columns={'score':'score_a','y_true':'y_a','year':'year_a'}
    )
    b = subset.loc[subset.feature_group.eq(group_b), ['date','year','y_true','score']].rename(
        columns={'score':'score_b','y_true':'y_b','year':'year_b'}
    )
    if a.empty or b.empty:
        return pd.DataFrame()
    out = a.merge(b, on='date', how='inner', validate='one_to_one').sort_values('date').reset_index(drop=True)
    if out.empty:
        return out
    if not out.y_a.eq(out.y_b).all() or not out.year_a.eq(out.year_b).all():
        raise ValueError(f'Różne etykiety lub lata w parze {model}/{drop}/{horizon}/{group_a}/{group_b}')
    if len(out) < len(a) or len(out) < len(b):
        warnings.warn(
            f'Porównanie {model}, {drop}/{horizon}: tylko wspólne daty ({len(out)}/{len(a)}/{len(b)}).'
        )
    return pd.DataFrame({
        'date':out.date, 'year':out.year_a, 'y_true':out.y_a,
        'score_a':out.score_a, 'score_b':out.score_b
    })


def ap_if_defined(y, scores, min_positives=MIN_POSITIVES_FOR_AP):
    y = np.asarray(y, dtype=int)
    if (y == 1).sum() < min_positives or (y == 0).sum() == 0:
        return np.nan
    return float(average_precision_score(y, scores))


def one_paired_comparison(pair):
    y = pair.y_true.to_numpy(dtype=int)
    pa, pb = pair.score_a.to_numpy(float), pair.score_b.to_numpy(float)
    ap_a = ap_if_defined(y, pa)
    ap_b = ap_if_defined(y, pb)
    prevalence = float(y.mean())
    return {
        'n_days':int(len(y)), 'positive_days':int(y.sum()),
        'positive_rate':prevalence,
        'AP_A':ap_a, 'AP_B':ap_b,
        'AP_A_minus_prevalence':ap_a-prevalence,
        'AP_B_minus_prevalence':ap_b-prevalence,
        'delta_AP_A_minus_B':ap_a-ap_b,
    }


def sample_indices_by_year(pair, block_size, rng):
    # Bloki kołowe chronią przed problemem ostatniego niedopełnionego bloku.
    sampled = []
    for year, grp in pair.groupby('year', sort=True):
        indices = grp.index.to_numpy(dtype=int)
        n = len(indices)
        if n == 0:
            continue
        out = []
        while len(out) < n:
            start = int(rng.integers(0, n))
            take = indices[(start + np.arange(block_size)) % n]
            out.extend(take.tolist())
        sampled.extend(out[:n])
    return np.asarray(sampled, dtype=int)


def paired_block_bootstrap(pair, block_size=60, n_boot=600, seed=42):
    pair = pair.sort_values(['year','date']).reset_index(drop=True)
    if pair.empty:
        return dict(ci_low=np.nan,ci_high=np.nan,boot_valid=0,boot_reps=n_boot,
                    probability_delta_positive=np.nan)
    rng = np.random.default_rng(seed)
    y = pair['y_true'].to_numpy(dtype=int)
    a = pair['score_a'].to_numpy(dtype=float)
    b = pair['score_b'].to_numpy(dtype=float)
    deltas = []
    for _ in range(n_boot):
        idx = sample_indices_by_year(pair, block_size, rng)
        if (y[idx] == 1).sum() < MIN_POSITIVES_FOR_AP or (y[idx] == 0).sum() == 0:
            continue
        deltas.append(
            average_precision_score(y[idx], a[idx]) -
            average_precision_score(y[idx], b[idx])
        )
    if len(deltas) < max(30, int(0.75*n_boot)):
        warnings.warn('Wiele próbek bootstrap bez dwóch klas; przedział może być niestabilny.')
    if not deltas:
        return dict(ci_low=np.nan,ci_high=np.nan,boot_valid=0,
                    boot_reps=n_boot, probability_delta_positive=np.nan)
    lo, hi = np.quantile(deltas, [0.025, 0.975])
    return dict(ci_low=float(lo), ci_high=float(hi), boot_valid=len(deltas),
                boot_reps=n_boot,
                probability_delta_positive=float(np.mean(np.asarray(deltas)>0)))

## 3. Sparowane porównania AP: LPPLS-only, Price-only i Price+LPPLS

Kluczowe porównanie artykułu to **`Price+LPPLS - Price-only`** na identycznych datach. Raportujemy także `LPPLS-only - Price-only`. Dla części 1 modele **nie są ponownie trenowane**.

In [ ]:
COMPARISONS = [
    ('Price+LPPLS', 'Price-only'),
    ('LPPLS-only', 'Price-only'),
]
paired_rows, annual_rows, sensitivity_rows = [], [], []
all_configs = sorted(set(zip(preds.drop_pct, preds.horizon)))
all_ml_models = sorted(set(preds.loc[preds.feature_group.ne('Baseline'), 'model']))
for d, h in all_configs:
    for model in all_ml_models:
        for ga, gb in COMPARISONS:
            pair = matched_predictions(preds, d, h, model, ga, gb)
            if pair.empty:
                continue
            stats = one_paired_comparison(pair)
            if not np.isfinite(stats['delta_AP_A_minus_B']):
                continue
            ci = paired_block_bootstrap(pair, block_size=BLOCK_SIZE,
                                        n_boot=BOOTSTRAP_REPS,
                                        seed=SEED + 100*d + h + all_ml_models.index(model))
            paired_rows.append(dict(drop_pct=d,horizon=h,model=model,
                                   group_A=ga,group_B=gb,**stats,**ci))
            for yr, part in pair.groupby('year'):
                yr_stats=one_paired_comparison(part)
                annual_rows.append(dict(drop_pct=d,horizon=h,year=int(yr),
                                        model=model,group_A=ga,group_B=gb,**yr_stats))
            # Analiza wrażliwości długości bloku dla podstawowego porównania.
            if ga=='Price+LPPLS':
                for bsz in BLOCK_SENSITIVITY:
                    ci_b=paired_block_bootstrap(pair, block_size=bsz,
                        n_boot=BOOTSTRAP_REPS, seed=SEED + d + h + bsz + all_ml_models.index(model))
                    sensitivity_rows.append(dict(drop_pct=d,horizon=h,model=model,
                                                 block_days=bsz,**ci_b,
                                                 observed_delta_AP=stats['delta_AP_A_minus_B']))

paired_summary=pd.DataFrame(paired_rows)
annual=pd.DataFrame(annual_rows)
sensitivity=pd.DataFrame(sensitivity_rows)
if paired_summary.empty:
    raise RuntimeError('Nie można porównać modeli: sprawdź dostępność wszystkich grup w 08b_04_predykcje_OOS.csv.')
paired_summary.to_csv(OUT/'09_01_paired_AP_blockbootstrap.csv',index=False)
annual.to_csv(OUT/'09_02_yearly_paired_AP.csv',index=False)
sensitivity.to_csv(OUT/'09_03_block_size_sensitivity.csv',index=False)
print('Przybliżone przedziały percentile-bootstrap; nie korygują wielokrotnych porównań.')
display(paired_summary.loc[paired_summary.group_A.eq('Price+LPPLS'),
        ['drop_pct','horizon','model','positive_rate','AP_A','AP_B',
         'delta_AP_A_minus_B','ci_low','ci_high','boot_valid']].round(4))

In [ ]:
# Wykres sparowanych różnic AP dla jednej konfiguracji.
plot_d, plot_h = (10,60) if (10,60) in all_configs else all_configs[0]
chart_data = paired_summary.loc[
    (paired_summary.drop_pct==plot_d) & (paired_summary.horizon==plot_h)
    & paired_summary.group_A.eq('Price+LPPLS')
].sort_values('delta_AP_A_minus_B')
if not chart_data.empty:
    fig, ax=plt.subplots(figsize=(10,5))
    center=chart_data['delta_AP_A_minus_B'].to_numpy()
    err=np.vstack([center-chart_data.ci_low.to_numpy(),
                   chart_data.ci_high.to_numpy()-center])
    # Bootstrap percentile może mieć skośność — rysujemy CI odrębnie, nie errorbar jeśli nie obejmuje estymatora.
    yy=np.arange(len(chart_data))
    ax.scatter(center, yy, label='Różnica AP (hybryda - ceny)')
    for i,(_,r) in enumerate(chart_data.iterrows()):
        ax.plot([r.ci_low,r.ci_high],[i,i],lw=2)
    ax.axvline(0,color='black',ls='--',lw=1)
    ax.set_yticks(yy,chart_data.model.tolist())
    ax.set_xlabel('ΔAP = AP(Price+LPPLS) − AP(Price-only)')
    ax.set_title(f'Sparowana różnica Average Precision — spadek {plot_d}% / {plot_h} dni')
    ax.grid(axis='x',alpha=.3)
    plt.tight_layout()
    plt.savefig(OUT/f'09_04_delta_AP_{plot_d}_{plot_h}.png',dpi=180)
    plt.show()

## 4. Ablation study — pięć *zagnieżdżonych* zestawów cech

- **A:** tylko historyczne zmiany cen i zmienność.
- **B:** A + LPPLS Confidence Indicator.
- **C:** B + podstawowe parametry LPPLS (`m`, `ω`, `t_gap`).
- **D:** C + `R²`.
- **E:** A + wszystkie cechy LPPLS używane w 08b.

Reużywamy **jednakowych hiperparametrów** wybranych wcześniej dla zestawu `Price+LPPLS` w danym modelu, roku i zadaniu. To kontrolowane porównanie cech przy stałym modelu; **nie** dowodzi, że inne zestawy cech nie mogłyby osiągnąć lepszych wyników po własnym strojeniu. Wybieramy próg na walidacji (rok `Y−1`), a do testu nie zaglądamy.

In [ ]:
PRICE_FEATURES = (['ret_1d']
    + [f'momentum_{n}d' for n in (7,14,30,60)]
    + [f'volatility_{n}d' for n in (7,30,60)]
    + [f'drawdown_{n}d' for n in (30,90)])
ALL_LPPLS = ['lppls_ci', 'lppls_n_accepted', 'lppls_m', 'lppls_w',
             'lppls_tgap', 'lppls_r2', 'lppls_B', 'lppls_C',
             'lppls_window_n', 'lppls_tgap_std']
ABLATION_GROUPS = {
    'A_Price':PRICE_FEATURES,
    'B_Price_CI':PRICE_FEATURES+['lppls_ci'],
    'C_Price_CI_m_w_tgap':PRICE_FEATURES+['lppls_ci','lppls_m','lppls_w','lppls_tgap'],
    'D_Price_CI_m_w_tgap_r2':PRICE_FEATURES+['lppls_ci','lppls_m','lppls_w','lppls_tgap','lppls_r2'],
    'E_Price_AllLPPLS':PRICE_FEATURES+ALL_LPPLS,
}


def make_model(model_name, params):
    if model_name=='LogisticRegression':
        return LogisticRegression(max_iter=2500,random_state=SEED,**params)
    if model_name=='DecisionTree':
        return DecisionTreeClassifier(random_state=SEED,**params)
    if model_name=='XGBoost':
        from xgboost import XGBClassifier
        return XGBClassifier(random_state=SEED,n_jobs=2,tree_method='hist',eval_metric='logloss',**params)
    if model_name=='LightGBM':
        from lightgbm import LGBMClassifier
        return LGBMClassifier(random_state=SEED,n_jobs=2,verbosity=-1,**params)
    if model_name=='CatBoost':
        from catboost import CatBoostClassifier
        return CatBoostClassifier(random_seed=SEED,thread_count=2,verbose=False,
                                  allow_writing_files=False,**params)
    raise ValueError('Nieznany model: '+model_name)


def fit_predict_pair(model_name, params, train, validation, test, features, target):
    # Imputer jest dopasowywany WYŁĄCZNIE na okresie treningowym.
    imputer=SimpleImputer(strategy='median',add_indicator=True,keep_empty_features=True)
    x_train=imputer.fit_transform(train[features]); x_val=imputer.transform(validation[features]); x_test=imputer.transform(test[features])
    if model_name=='LogisticRegression':
        scaler=StandardScaler()
        x_train=scaler.fit_transform(x_train)
        x_val=scaler.transform(x_val)
        x_test=scaler.transform(x_test)
    model=make_model(model_name,params)
    model.fit(x_train,train[target].to_numpy(dtype=int))
    return model.predict_proba(x_val)[:,1], model.predict_proba(x_test)[:,1]


def threshold_f1(y, score):
    y=np.asarray(y,dtype=int)
    if len(np.unique(y)) < 2:
        return 0.5
    candidates=[]
    for t in PROB_THRESHOLDS:
        pred=(score>=t).astype(int)
        candidates.append((f1_score(y,pred,zero_division=0), -int(pred.sum()), float(t)))
    return max(candidates,key=lambda r:(r[0],r[1],r[2]))[2]

In [ ]:
if RUN_ABLATION:
    daily_path = locate('01_cechy_dzienne.csv')
    selected_path = locate('08b_02_wybrane_hiperparametry.csv')
    data = pd.read_csv(daily_path, parse_dates=['date'])
    selected = pd.read_csv(selected_path)
    data['date']=pd.to_datetime(data['date'], utc=True).dt.tz_convert(None).dt.normalize()
    assert data['date'].is_unique and data['date'].is_monotonic_increasing
    missing_features=set(sum(ABLATION_GROUPS.values(),[]))-set(data.columns)
    if missing_features:
        raise ValueError(f'Brak cech w 08b_01_cechy_dzienne: {sorted(missing_features)}')
    for col in sum(ABLATION_GROUPS.values(),[]):
        data[col]=pd.to_numeric(data[col],errors='coerce')
    require_sel={'year','drop_pct','horizon','model','feature_group','params'}
    if not require_sel.issubset(selected.columns):
        raise ValueError('Niepełny 08b_02_wybrane_hiperparametry.csv: '+str(require_sel-set(selected.columns)))
    print('Dane do ablacji:', data.shape, 'wybrane parametry:', selected.shape)

## 5. Trening ablation z tym samym schematem czasowym co 08b

Dla testu `Y`: trening `date < Y−1` z `label_end < Y−1`, walidacja progowa w `Y−1` z dostępnym wynikiem przed `Y`, test = rok `Y`. To zapobiega wykorzystaniu etykiet, które stały się znane dopiero po decyzji.

In [ ]:
ablation_preds=[]
ablation_folds=[]
ablation_skips=[]
if RUN_ABLATION:
    for drop,horizon in ABLATION_TARGETS:
        target=f'drop_{drop}_h{horizon}'
        end_col=f'label_end_h{horizon}'
        if not {target,end_col}.issubset(data.columns):
            raise ValueError(f'Brak {target} lub {end_col}: wykonaj 08b RUN_ALL_NINE=True i notatnik 07.')
        sample=data.loc[data[target].notna()].copy()
        sample[target]=pd.to_numeric(sample[target],errors='raise').astype(int)
        sample[end_col]=pd.to_datetime(sample[end_col],errors='coerce')
        assert sample[end_col].notna().all()
        for year in (2021,2022,2023,2024):
            val_start=pd.Timestamp(f'{year-1}-01-01'); test_start=pd.Timestamp(f'{year}-01-01'); test_end=pd.Timestamp(f'{year+1}-01-01')
            tr=sample.loc[(sample.date<val_start)&(sample[end_col]<val_start)]
            val=sample.loc[(sample.date>=val_start)&(sample.date<test_start)&(sample[end_col]<test_start)]
            test=sample.loc[(sample.date>=test_start)&(sample.date<test_end)]
            if tr.empty or val.empty or test.empty or tr[target].nunique()<2:
                ablation_skips.append((drop,horizon,year,'empty_or_single_class'))
                continue
            assert tr[end_col].max()<val.date.min()
            assert val[end_col].max()<test.date.min()
            for model_name in ABLATION_MODELS:
                rows=selected.loc[(selected['year']==year)&(selected['drop_pct']==drop)
                    &(selected['horizon']==horizon)&(selected['model']==model_name)
                    &(selected['feature_group']=='Price+LPPLS')]
                if len(rows)!=1:
                    ablation_skips.append((drop,horizon,year,model_name+'_missing_or_multiple_params'))
                    continue
                params=json.loads(rows.iloc[0]['params'])
                for group_name,feat in ABLATION_GROUPS.items():
                    try:
                        proba_val,proba_test=fit_predict_pair(model_name,params,tr,val,test,feat,target)
                        threshold=threshold_f1(val[target].to_numpy(dtype=int),proba_val)
                        pred_test=(proba_test>=threshold).astype(int)
                        yy=test[target].to_numpy(dtype=int)
                        tn,fp,fn,tp=confusion_matrix(yy,pred_test,labels=[0,1]).ravel()
                        ablation_folds.append(dict(drop_pct=drop,horizon=horizon,year=year,
                            model=model_name,feature_group=group_name,threshold=threshold,
                            n=len(yy), positives=int(yy.sum()),TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),
                            AP=ap_if_defined(yy,proba_test)))
                        ablation_preds.append(pd.DataFrame(dict(date=test.date.to_numpy(),year=year,
                            drop_pct=drop,horizon=horizon,model=model_name,feature_group=group_name,
                            y_true=yy,score=proba_test,y_pred=pred_test)))
                    except Exception as exc:
                        ablation_skips.append((drop,horizon,year,model_name,group_name,str(exc)))
                        warnings.warn(f'Błąd trenowania: {drop}/{horizon}/{year}/{model_name}/{group_name}: {exc}')
    if not ablation_preds:
        raise RuntimeError('Ablation nie zwróciła żadnych wyników. Sprawdź wybrane modele, zadania, pliki i liczbę klas.')
    ablation_predictions=pd.concat(ablation_preds,ignore_index=True)
    ablation_fold_results=pd.DataFrame(ablation_folds)
    ablation_predictions.to_csv(OUT/'09_05_ablation_predictions_OOS.csv',index=False)
    ablation_fold_results.to_csv(OUT/'09_06_ablation_yearly.csv',index=False)
    pd.DataFrame({'details':[str(x) for x in ablation_skips]}).to_csv(OUT/'09_07_ablation_skips.csv',index=False)
    print('Prognozy ablation:',len(ablation_predictions),'| liczba testów:',len(ablation_fold_results))
    if ablation_skips:
        print('Pominięte przypadki:',len(ablation_skips),'- zobacz 09_07_ablation_skips.csv')

## 6. Zagregowane metryki i krokowe ΔAP

Podstawowa hipoteza ablation: po dodaniu cech LPPLS rośnie AP w tym samym okresie OOS. Każde porównanie obliczamy na **sparowanych datach**. Nie traktujemy najwyższego AP spośród wielu modeli jako bezstronnej prognozy testowej.

In [ ]:
if RUN_ABLATION:
    ab_rows=[]
    for (d,h,model,group),part in ablation_predictions.groupby(
        ['drop_pct','horizon','model','feature_group'],sort=False
    ):
        y=part['y_true'].to_numpy(dtype=int)
        yhat=part['y_pred'].to_numpy(dtype=int)
        tn,fp,fn,tp=confusion_matrix(y,yhat,labels=[0,1]).ravel()
        ab_rows.append(dict(drop_pct=d,horizon=h,model=model,feature_group=group,
            n_days=len(y),positive_rate=float(y.mean()),
            AP=ap_if_defined(y,part['score'].to_numpy(float)),
            F1=(2*tp/(2*tp+fp+fn) if (2*tp+fp+fn) else np.nan),
            TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn)))
    ablation_summary=pd.DataFrame(ab_rows)
    ablation_summary['AP_minus_prevalence']=ablation_summary.AP-ablation_summary.positive_rate
    ablation_summary.to_csv(OUT/'09_08_ablation_AP_summary.csv',index=False)
    display(ablation_summary.round(4))

    ablation_steps=[
        ('B_Price_CI','A_Price'),
        ('C_Price_CI_m_w_tgap','B_Price_CI'),
        ('D_Price_CI_m_w_tgap_r2','C_Price_CI_m_w_tgap'),
        ('E_Price_AllLPPLS','D_Price_CI_m_w_tgap_r2'),
        ('E_Price_AllLPPLS','A_Price'),
    ]
    steps=[]
    for d,h in ABLATION_TARGETS:
        for model in ABLATION_MODELS:
            for ga,gb in ablation_steps:
                pair=matched_predictions(ablation_predictions,d,h,model,ga,gb)
                if pair.empty:
                    continue
                stats=one_paired_comparison(pair)
                ci=paired_block_bootstrap(pair,block_size=BLOCK_SIZE,
                        n_boot=BOOTSTRAP_REPS,seed=SEED+d+h+len(steps))
                steps.append(dict(drop_pct=d,horizon=h,model=model,group_A=ga,group_B=gb,
                                  **stats,**ci))
    ablation_step_summary=pd.DataFrame(steps)
    ablation_step_summary.to_csv(OUT/'09_09_ablation_step_deltas.csv',index=False)
    display(ablation_step_summary[['drop_pct','horizon','model','group_A','group_B',
        'delta_AP_A_minus_B','ci_low','ci_high','n_days']].round(4))

In [ ]:
# Wykres ablation (dla każdej konfiguracji, osobny wykres)
if RUN_ABLATION:
    for d,h in ABLATION_TARGETS:
        sub=ablation_summary.loc[(ablation_summary.drop_pct==d)&(ablation_summary.horizon==h)]
        piv=sub.pivot(index='feature_group',columns='model',values='AP').reindex(list(ABLATION_GROUPS))
        ax=piv.plot(marker='o',figsize=(11,5))
        ax.set_title(f'Ablation LPPLS — spadek BTC ≥{d}% / {h} dni (OOS)')
        ax.set_ylabel('Average Precision')
        ax.set_xlabel('Zestaw cech')
        ax.set_xticks(range(len(piv.index)),['A','B','C','D','E'])
        ax.grid(alpha=.3)
        plt.tight_layout()
        plt.savefig(OUT/f'09_10_ablation_AP_{d}_{h}.png',dpi=180)
        plt.show()

## 7. Wnioski i reguły raportowania

1. **Oceń ΔAP, nie tylko najwyższy słupek.** Jeśli 95% bootstrap CI zawiera zero, nie przedstawiaj różnicy jako przekonująco potwierdzonej.
2. **Raportuj częstość klasy pozytywnej** (`positive_rate`) obok AP; przy wysokim prevalence AP może wyglądać dobrze nawet bez silnego modelu.
3. **Podawaj wyniki osobno dla lat**, a także połączony OOS. Lata 2021–2024 były już oglądane; nie jest to całkowicie niezależny eksperyment potwierdzający.
4. **Bloki 30/60/90 dni** dają analizę wrażliwości; silna autokorelacja i mała liczba epizodów ograniczają wiarygodność nominalnego 95% CI.
5. **Unikaj narracji o istotności na podstawie jednego korzystnego testu.** Dziewięć etykiet oraz wiele algorytmów tworzą problem wielokrotnych porównań; wyniki wymagają dalszej walidacji na nowych danych/aktywach.
6. **F1 i alarmy** służą do oceny reguły progowej; AP ocenia ranking bez wybranego progu. Do publikacji dodaj liczbę niezależnych alarmów i wyprzedzenie zdarzeń, aby wykazać sens systemu wczesnego ostrzegania.

Gotowe pliki w `wyniki_ablation_09`. Nie kopiuj tu do zmiennych wejściowych `literature_bubble`, `future_drop` ani dat `label_end`.